# gacha_gacha — ガチャシステムでデータベースとバックエンドを動かしながら学ぶ

本 Notebook は **Colab で上から順に実行する** だけで、ガチャゲームのバックエンド一式が
組み上がり、ノートブックの中で実際にガチャを回せるようになる学習教材です。

## ねらい

- HTTP サーバー / SQL / Python ガチャ抽選 を、フレームワークの魔法に頼らず
  「自分で書く感覚」 で身につける
- AI に書かせて読むだけだと身につかない → **コピペでも手書きでも、実際にコードが動く** 状態を経験する
- 章を進めるごとに、最終的なフルセット (`server/app.py` + `db/schema.sql` + `web/index.html`)
  が、目の前で 1 ファイルずつ仕上がっていく

## 対象

- プログラミング経験 1〜2 年 (Java / Kotlin など)
- Python は構文が読める程度で OK
- データベースは PK/FK を聞いた程度の前提

## 進め方

1. **上から順に実行** (Shift+Enter)。 各セルの出力を見ながら進む
2. 気が向いたらセルを書き換えて壊してみる。動かなくなったら戻して再実行
3. 最終セクションでガチャ画面が Notebook 内に現れたら完走

## DB について

この Notebook は 学習を最短で始めるため **SQLite** を使います (Python 標準ライブラリ、追加インストール不要)。
本格的な PostgreSQL 環境はリポジトリ直下の `docker-compose.yml` に用意してあり、
コードを変えずに切り替え可能です (最終章で説明)。


---
## 第 0 章 — 環境セットアップ

このセクションを実行すると、Notebook が動いている場所の下に
`gacha_gacha/` という作業フォルダができ、そこに章ごとのファイルを書き込んでいきます。

> Colab なら `/content/gacha_gacha` に作られます。ローカルで実行している場合は
> 現在のディレクトリ直下に作られます。


In [ ]:
import os, pathlib, sys

# Colab かどうかで作業ディレクトリを変える
if pathlib.Path("/content").exists():
    ROOT = pathlib.Path("/content/gacha_gacha")
else:
    ROOT = pathlib.Path.cwd() / "gacha_gacha_workspace"
ROOT.mkdir(exist_ok=True, parents=True)

# 必要なサブディレクトリを作る
for sub in ["db", "server", "web", "exercises/ch01", "exercises/ch02"]:
    (ROOT / sub).mkdir(exist_ok=True, parents=True)

# 以降のセルから相対パスで使えるように移動
os.chdir(ROOT)
print("作業ディレクトリ:", ROOT)
print("中身:", sorted(p.name for p in ROOT.iterdir()))

# server/ をモジュールとして import できるようにパスを通す
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


---
# 第 1 章 — HTTPサーバーを建ててみよう

最終的にはガチャの API になりますが、いきなり API は建てません。
**1 番下の TCP ソケットから始めて、6 ステップで API の足場を組みます**。

最後 (Step 6) のコードは、本物の `server/app.py` の骨格とほぼ同じになります。

## HTTP は「一往復のテキスト」

クライアントが手紙を 1 通送り、サーバーが返事を 1 通返す。
手紙の中身はただのテキスト。実物はこんな形:

```
POST /api/login HTTP/1.1
Host: localhost:8000
Content-Type: application/json
Content-Length: 39

{"name":"alice","password":"secret"}
```

```
HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Content-Length: 24

{"user":{"id":1,"name":"alice"}}
```

これだけです。ヘッダ + 空行 + 本文。Web の上で動いているもの全て、この往復の上に積み上がっています。


## Step 1 — TCP ソケットだけで HTTP風のテキストを返す

HTTP の前提として、TCP ソケットがバイト列を運んでいます。
Python の `socket` モジュールを使うと、その層から書けます。

下のセルを実行すると、ファイル `exercises/ch01/step1_socket.py` ができます。
そのあと、もう 1 つ下のセルでバックグラウンド起動 → curl 相当の確認まで自動で行います。


In [ ]:
%%writefile exercises/ch01/step1_socket.py
"""ch01 / step 1 — TCPソケットだけでHTTP風の返事を返す。

ゴール:
    HTTP は「テキスト」「TCP の上」「往復1セット」というのを、
    フレームワークなしで体感する。

実行:
    python exercises/ch01/step1_socket.py     (= ターミナル A)
    別ターミナル B で:
        curl -v http://127.0.0.1:9000/

何が起きるか:
    - 9000 番ポートでTCP接続を待つ
    - 1 件接続が来たら、最大 4096 バイト読み取って中身を表示
    - "HTTP/1.1 200 OK ..." という形のテキストを返して切断

確認ポイント:
    1. curl の出力に > GET / HTTP/1.1, > Host: ... のような送信ヘッダが見える
    2. ターミナル A 側に、curl が送ってきたリクエスト全文 (= ただのテキスト) が出る
    3. < HTTP/1.1 200 OK が curl 側に表示される
"""

import socket


def main() -> None:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    # 同じポートですぐ再起動できるようにする小ワザ
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    sock.bind(("127.0.0.1", 9000))
    sock.listen()
    print("listening on http://127.0.0.1:9000  (Ctrl+C で終了)")

    try:
        while True:
            conn, addr = sock.accept()
            data = conn.recv(4096)
            print("---- received ----")
            print(data.decode(errors="replace"))
            print("---- end -----")

            body = b"hello, raw tcp\n"
            response = (
                b"HTTP/1.1 200 OK\r\n"
                b"Content-Type: text/plain; charset=utf-8\r\n"
                b"Content-Length: " + str(len(body)).encode() + b"\r\n"
                b"\r\n"
                + body
            )
            conn.sendall(response)
            conn.close()
    except KeyboardInterrupt:
        print("\nbye")


if __name__ == "__main__":
    main()


In [ ]:
# Step 1 を別スレッドで起動して、内部から確認する
import socket, threading, urllib.request, time

def step1_serve_once():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("127.0.0.1", 9000))
    s.listen()
    conn, _ = s.accept()
    raw = conn.recv(4096)
    print("---- received ----")
    print(raw.decode(errors="replace"))
    print("---- end ----")
    body = b"hello, raw tcp\n"
    conn.sendall(b"HTTP/1.1 200 OK\r\n"
                 b"Content-Type: text/plain; charset=utf-8\r\n"
                 b"Content-Length: " + str(len(body)).encode() + b"\r\n"
                 b"\r\n" + body)
    conn.close()
    s.close()

t = threading.Thread(target=step1_serve_once, daemon=True)
t.start()
time.sleep(0.1)
resp = urllib.request.urlopen("http://127.0.0.1:9000/")
print("status:", resp.status)
print("body  :", resp.read().decode())
t.join(timeout=1)


**期待される出力 (抜粋):**
```
---- received ----
GET / HTTP/1.1
Accept-Encoding: identity
Host: 127.0.0.1:9000
...
---- end ----
status: 200
body  : hello, raw tcp
```

### 観察ポイント
- HTTP は「テキスト」であること (リクエストの中身が普通の英語+改行)
- `Content-Length` は応答ボディの**バイト数**。これが嘘だとクライアントが固まる
- ヘッダの後ろに**空行 (\r\n\r\n)** が必ず必要

### 宿題 (やってみよう)
1. `Content-Type` を `text/html` に変えて `<h1>hello</h1>` を返してみる
2. ステータスコードを `418` (I'm a teapot) にしてみる


## Step 2 — `http.server` で最小の Hello GET

Step 1 のように毎回ヘッダを手で書くのは辛い。
Python 標準の `http.server.BaseHTTPRequestHandler` に乗り換えます。
`do_GET` をオーバーライドすると GET 全般が捕まえられます。


In [ ]:
%%writefile exercises/ch01/step2_http_server_min.py
"""ch01 / step 2 — http.server で最小の Hello サーバー。

ゴール:
    ヘッダのパースを毎回手書きしなくて済むよう、Python標準の
    http.server に乗り換える。BaseHTTPRequestHandler が do_GET を
    呼んでくれることを確認する。

実行:
    python exercises/ch01/step2_http_server_min.py
    curl -v http://127.0.0.1:8001/
    curl -v http://127.0.0.1:8001/anything

宿題:
    1. send_header("Content-Type", "...") の値を text/html に変えて、
       <h1>hello</h1> を返してみよう。ブラウザで開くと太字になる?
    2. self.send_response(200) を 418 に変えると、curl は何と表示する?
"""

from http.server import BaseHTTPRequestHandler, HTTPServer


class Hello(BaseHTTPRequestHandler):
    def do_GET(self):  # noqa: N802
        body = b"hello, http\n"
        self.send_response(200)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


def main() -> None:
    addr = ("127.0.0.1", 8001)
    print(f"listening on http://{addr[0]}:{addr[1]}")
    HTTPServer(addr, Hello).serve_forever()


if __name__ == "__main__":
    main()


In [ ]:
# Step 2 を別スレッドで起動 → 内部から GET して確認
from http.server import BaseHTTPRequestHandler, HTTPServer
import threading, urllib.request, time

class Hello(BaseHTTPRequestHandler):
    def do_GET(self):
        body = b"hello, http\n"
        self.send_response(200)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, fmt, *args): pass  # quiet

server = HTTPServer(("127.0.0.1", 8001), Hello)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(0.1)
for path in ["/", "/anything", "/api/foo"]:
    r = urllib.request.urlopen("http://127.0.0.1:8001" + path)
    print(path, "->", r.status, r.read())
server.shutdown()


**期待される出力:**
```
/ -> 200 b'hello, http\n'
/anything -> 200 b'hello, http\n'
/api/foo -> 200 b'hello, http\n'
```

どのパスでも同じ応答が返るのに気付きましたか?
パスごとに分けるのが Step 3 です。


## Step 3 — パスで分岐 + クエリ文字列を受け取る

`self.path` は `"/api/echo?msg=hi"` のような形で来るので、
`urllib.parse.urlparse` でパスとクエリに分けます。


In [ ]:
%%writefile exercises/ch01/step3_routing.py
"""ch01 / step 3 — パスごとに違う応答を返す (= "ルーティング"の素朴版)。

ゴール:
    self.path で分岐するという素朴な実装を体験する。
    クエリ文字列 (?msg=...) のパースも urllib.parse でやってみる。

実装する API:
    GET /              -> "hello, http\\n"
    GET /api/health    -> JSON {"status": "ok"}
    GET /api/echo?msg=X -> JSON {"you_said": "X"}
    その他              -> 404 + JSON {"error": "Not Found"}

実行:
    python exercises/ch01/step3_routing.py
    curl http://127.0.0.1:8001/
    curl http://127.0.0.1:8001/api/health
    curl 'http://127.0.0.1:8001/api/echo?msg=hi%20there'
    curl -i http://127.0.0.1:8001/nope    # -i でステータス行も表示

宿題:
    1. /api/echo に msg が無いとき、現在は空文字を返す。400 を返すように変えてみよう
    2. /api/time を増やして、現在時刻 (datetime.utcnow().isoformat()) を返してみよう
"""

import json
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs


class App(BaseHTTPRequestHandler):
    def do_GET(self):  # noqa: N802
        url = urlparse(self.path)
        path = url.path
        query = parse_qs(url.query)

        if path == "/":
            self._text(200, "hello, http\n")
        elif path == "/api/health":
            self._json(200, {"status": "ok"})
        elif path == "/api/echo":
            msg = query.get("msg", [""])[0]
            self._json(200, {"you_said": msg})
        else:
            self._json(404, {"error": "Not Found", "path": path})

    # -- helpers ----------------------------------------------------------
    def _text(self, status: int, s: str) -> None:
        body = s.encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _json(self, status: int, obj: dict) -> None:
        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


def main() -> None:
    addr = ("127.0.0.1", 8001)
    print(f"listening on http://{addr[0]}:{addr[1]}")
    HTTPServer(addr, App).serve_forever()


if __name__ == "__main__":
    main()


In [ ]:
# Step 3 を起動して 4 通りのパスを叩く
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse, parse_qs
import json, threading, urllib.request, time

class App(BaseHTTPRequestHandler):
    def do_GET(self):
        url = urlparse(self.path); path = url.path; q = parse_qs(url.query)
        if path == "/":              return self._text(200, "hello, http\n")
        if path == "/api/health":    return self._json(200, {"status": "ok"})
        if path == "/api/echo":      return self._json(200, {"you_said": q.get("msg", [""])[0]})
        return self._json(404, {"error": "Not Found", "path": path})
    def _text(self, s, body):
        b = body.encode(); self.send_response(s)
        self.send_header("Content-Type", "text/plain; charset=utf-8")
        self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
    def _json(self, s, obj):
        b = json.dumps(obj, ensure_ascii=False).encode(); self.send_response(s)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
    def log_message(self, fmt, *args): pass

server = HTTPServer(("127.0.0.1", 8002), App)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(0.1)
for path in ["/", "/api/health", "/api/echo?msg=hello%20world", "/missing"]:
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8002" + path)
        print(f"{path:35s} -> {r.status} {r.read().decode()[:60]}")
    except urllib.error.HTTPError as e:
        print(f"{path:35s} -> {e.code} {e.read().decode()[:60]}")
server.shutdown()


**期待される出力:**
```
/                                   -> 200 hello, http
/api/health                         -> 200 {"status": "ok"}
/api/echo?msg=hello%20world         -> 200 {"you_said": "hello world"}
/missing                            -> 404 {"error": "Not Found", "path": "/missing"}
```


## Step 4 — POST で JSON を受け取る

ガチャを引く API は POST します。
本文の長さは `Content-Length` ヘッダから読み取り、`self.rfile.read(n)` でバイト列を取得して JSON にデコードします。


In [ ]:
%%writefile exercises/ch01/step4_post_json.py
"""ch01 / step 4 — POST で JSON を受け取り、加工して返す。

ゴール:
    本物のWeb APIで頻出する「ボディからJSONを読む」を覚える。
    Content-Length / rfile / json.loads / json.dumps の往復を完全に書ける状態を目指す。

実装する API:
    POST /api/echo
        body: {"name": "alice", "age": 7}
        resp: {"received": {...}, "size": 23}
    POST /api/sum
        body: {"a": 1, "b": 2}
        resp: {"answer": 3}
    どちらも JSON 不正なら 400, 必須キー欠損なら 400, それ以外は 200。

実行:
    python exercises/ch01/step4_post_json.py
    curl -X POST http://127.0.0.1:8001/api/echo \\
         -H "Content-Type: application/json" \\
         -d '{"name":"alice","age":7}'
    curl -X POST http://127.0.0.1:8001/api/sum \\
         -H "Content-Type: application/json" \\
         -d '{"a":1,"b":2}'

宿題:
    1. /api/sum で a, b の片方が文字列だったら 400 を返すよう厳しくしよう
    2. /api/echo の応答に "method": "POST" を追加してみよう
"""

import json
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse


class App(BaseHTTPRequestHandler):
    def do_POST(self):  # noqa: N802
        path = urlparse(self.path).path
        try:
            body = self._read_json()
        except ValueError as e:
            return self._json(400, {"error": f"invalid json: {e}"})

        if path == "/api/echo":
            return self._json(200, {"received": body,
                                    "size": len(json.dumps(body).encode())})
        if path == "/api/sum":
            try:
                a, b = body["a"], body["b"]
            except KeyError:
                return self._json(400, {"error": "a と b が必要"})
            return self._json(200, {"answer": a + b})

        return self._json(404, {"error": "Not Found"})

    # ---- helpers --------------------------------------------------------
    def _read_json(self) -> dict:
        n = int(self.headers.get("Content-Length", "0"))
        raw = self.rfile.read(n)
        if not raw:
            return {}
        return json.loads(raw.decode("utf-8"))

    def _json(self, status: int, obj: dict) -> None:
        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


def main() -> None:
    addr = ("127.0.0.1", 8001)
    print(f"listening on http://{addr[0]}:{addr[1]}")
    HTTPServer(addr, App).serve_forever()


if __name__ == "__main__":
    main()


In [ ]:
# Step 4 を起動して /api/sum に POST してみる
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse
import json, threading, urllib.request, time

class App(BaseHTTPRequestHandler):
    def do_POST(self):
        path = urlparse(self.path).path
        n = int(self.headers.get("Content-Length", "0"))
        try:
            body = json.loads(self.rfile.read(n).decode("utf-8")) if n else {}
        except json.JSONDecodeError as e:
            return self._json(400, {"error": f"invalid json: {e}"})
        if path == "/api/echo":
            return self._json(200, {"received": body, "size": n})
        if path == "/api/sum":
            try: a, b = body["a"], body["b"]
            except KeyError: return self._json(400, {"error": "a と b が必要"})
            return self._json(200, {"answer": a + b})
        return self._json(404, {"error": "Not Found"})
    def _json(self, s, obj):
        b = json.dumps(obj, ensure_ascii=False).encode()
        self.send_response(s)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(b))); self.end_headers(); self.wfile.write(b)
    def log_message(self, fmt, *args): pass

server = HTTPServer(("127.0.0.1", 8003), App)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(0.1)

def post(path, payload):
    req = urllib.request.Request(
        "http://127.0.0.1:8003" + path,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST")
    try:
        return urllib.request.urlopen(req).read().decode()
    except urllib.error.HTTPError as e:
        return f"[{e.code}] " + e.read().decode()

print("echo:", post("/api/echo", {"name": "alice", "age": 7}))
print("sum :", post("/api/sum", {"a": 1, "b": 2}))
print("bad :", post("/api/sum", {"a": 1}))   # b 欠落で 400
server.shutdown()


**期待される出力:**
```
echo: {"received": {"name": "alice", "age": 7}, "size": 24}
sum : {"answer": 3}
bad : [400] {"error": "a と b が必要"}
```


## Step 5 — ルーター辞書でデータ駆動に

`do_GET` 内で if/elif の山を書くのは、ハンドラが増えると破綻します。
**`(method, path) → handler`** という辞書でディスパッチする形に書き換えます。
ここで初めて「フレームワークっぽいもの」を手作りした感覚になります。


In [ ]:
%%writefile exercises/ch01/step5_router_dict.py
"""ch01 / step 5 — ルーター辞書でディスパッチをデータ駆動にする。

ゴール:
    do_GET 内の if/elif の山をやめて、(method, path) -> handler の
    辞書でディスパッチする方式に書き換える。
    ここで初めて「フレームワークっぽいもの」を手作りする感覚に到達する。

実装する API:
    GET  /api/health -> {"status": "ok"}
    POST /api/echo   -> {"received": ..., "size": ...}
    POST /api/sum    -> {"answer": a+b}

実行:
    python exercises/ch01/step5_router_dict.py
    curl http://127.0.0.1:8001/api/health
    curl -X POST http://127.0.0.1:8001/api/echo \\
         -H 'Content-Type: application/json' -d '{"x":1}'

宿題:
    1. @route("DELETE", "/api/echo") を増やして、204 No Content を返すハンドラを書こう
    2. ルートが見つからない場合の 404 を、「存在するルート一覧」を含む JSON に拡張しよう
"""

import json
from http.server import BaseHTTPRequestHandler, HTTPServer
from typing import Callable
from urllib.parse import urlparse


# (method, path) -> handler
ROUTES: dict[tuple[str, str], Callable[["App"], None]] = {}


def route(method: str, path: str):
    def deco(fn):
        ROUTES[(method.upper(), path)] = fn
        return fn
    return deco


@route("GET", "/api/health")
def health(h: "App") -> None:
    h._json(200, {"status": "ok"})


@route("POST", "/api/echo")
def echo(h: "App") -> None:
    body = h._read_json()
    h._json(200, {"received": body,
                  "size": len(json.dumps(body).encode())})


@route("POST", "/api/sum")
def add(h: "App") -> None:
    body = h._read_json()
    h._json(200, {"answer": body["a"] + body["b"]})


class App(BaseHTTPRequestHandler):
    def do_GET(self):    self._dispatch("GET")     # noqa: N802
    def do_POST(self):   self._dispatch("POST")    # noqa: N802

    def _dispatch(self, method: str) -> None:
        path = urlparse(self.path).path
        handler = ROUTES.get((method, path))
        if handler is None:
            return self._json(404, {"error": "Not Found",
                                    "available": [
                                        f"{m} {p}" for (m, p) in sorted(ROUTES)
                                    ]})
        handler(self)

    # ---- helpers --------------------------------------------------------
    def _read_json(self) -> dict:
        n = int(self.headers.get("Content-Length", "0"))
        raw = self.rfile.read(n) if n else b""
        return json.loads(raw.decode("utf-8")) if raw else {}

    def _json(self, status: int, obj: dict) -> None:
        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


def main() -> None:
    addr = ("127.0.0.1", 8001)
    print(f"listening on http://{addr[0]}:{addr[1]}")
    print("  routes:")
    for (m, p) in sorted(ROUTES):
        print(f"    {m} {p}")
    HTTPServer(addr, App).serve_forever()


if __name__ == "__main__":
    main()


ポイント:
- `@route("GET", "/api/health")` というデコレータが `ROUTES` 辞書に登録する
- `do_GET` / `do_POST` は単に `_dispatch(method)` を呼ぶだけ
- `_dispatch` は `ROUTES.get((method, path))` でハンドラを取り出して呼ぶ
- このパターンは Flask, FastAPI, Express など、ありとあらゆる Web フレームワークの中身です


## Step 6 — 例外を JSON エラーに変換する

ハンドラ内で `raise bad_request("a と b は数値で")` と書くと、
**自動的に** 400 + `{"error": "..."}` が返る形を作ります。

予期せぬ例外は traceback を stderr に出して 500 を返す。
ここまで仕上がると、本物の `server/app.py` とほぼ同じ骨格になります。


In [ ]:
%%writefile exercises/ch01/step6_error_handling.py
"""ch01 / step 6 — 例外を「ステータス付きエラー」に変換する。

ゴール:
    ハンドラ側で `raise bad_request("...")` と書くと、自動的に
    400 + {"error": "..."} の JSON が返る形を作る。
    予期しないバグは 500 + traceback ログに流す。
    ここまでくると、サーバー本体 server/app.py の構造とほぼ同じになる。

実装する API:
    GET  /api/health
    POST /api/sum   { "a": <number>, "b": <number> }
        - a, b が無い -> 400
        - a, b が数値でない -> 400
        - 他は 200 で {"answer": a+b}

実行:
    python exercises/ch01/step6_error_handling.py
    curl -X POST http://127.0.0.1:8001/api/sum \\
         -H 'Content-Type: application/json' -d '{"a":1,"b":2}'
    curl -X POST http://127.0.0.1:8001/api/sum \\
         -H 'Content-Type: application/json' -d '{"a":"hi","b":2}'

宿題:
    1. unauthorized()  (= 401) ヘルパを作って、Cookie が無いときに投げてみよう
       (まだ cookie の値は使わなくて良い。ヘッダにそもそも Cookie が無いなら 401)
    2. server/app.py の AppError と _dispatch を比較してみよう。同じ作りになっているはず
"""

import json
import traceback
from http.server import BaseHTTPRequestHandler, HTTPServer
from typing import Callable
from urllib.parse import urlparse


class AppError(Exception):
    def __init__(self, status: int, message: str):
        super().__init__(message)
        self.status, self.message = status, message


def bad_request(msg: str) -> AppError: return AppError(400, msg)
def not_found(msg: str = "Not Found") -> AppError: return AppError(404, msg)


ROUTES: dict[tuple[str, str], Callable[["App"], None]] = {}


def route(method: str, path: str):
    def deco(fn):
        ROUTES[(method.upper(), path)] = fn
        return fn
    return deco


@route("GET", "/api/health")
def health(h: "App") -> None:
    h._json(200, {"status": "ok"})


@route("POST", "/api/sum")
def add(h: "App") -> None:
    body = h._read_json()
    if "a" not in body or "b" not in body:
        raise bad_request("a と b が必要")
    a, b = body["a"], body["b"]
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        raise bad_request("a と b は数値で")
    h._json(200, {"answer": a + b})


class App(BaseHTTPRequestHandler):
    def do_GET(self):    self._dispatch("GET")     # noqa: N802
    def do_POST(self):   self._dispatch("POST")    # noqa: N802

    def _dispatch(self, method: str) -> None:
        path = urlparse(self.path).path
        handler = ROUTES.get((method, path))
        if handler is None:
            return self._json(404, {"error": "Not Found", "path": path})
        try:
            handler(self)
        except AppError as e:
            self._json(e.status, {"error": e.message})
        except Exception as e:  # noqa: BLE001
            traceback.print_exc()
            self._json(500, {"error": "internal server error",
                             "detail": str(e)})

    def _read_json(self) -> dict:
        n = int(self.headers.get("Content-Length", "0"))
        raw = self.rfile.read(n) if n else b""
        try:
            return json.loads(raw.decode("utf-8")) if raw else {}
        except json.JSONDecodeError as e:
            raise bad_request(f"invalid json: {e}")

    def _json(self, status: int, obj: dict) -> None:
        body = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)


def main() -> None:
    addr = ("127.0.0.1", 8001)
    print(f"listening on http://{addr[0]}:{addr[1]}")
    HTTPServer(addr, App).serve_forever()


if __name__ == "__main__":
    main()


In [ ]:
# Step 6 を起動して、正常系と異常系を両方確認
from exercises.ch01.step6_error_handling import App
from http.server import HTTPServer
import threading, urllib.request, urllib.error, json, time

server = HTTPServer(("127.0.0.1", 8006), App)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(0.2)

def hit(method, path, body=None):
    req = urllib.request.Request(
        "http://127.0.0.1:8006" + path,
        data=json.dumps(body).encode() if body else None,
        headers={"Content-Type": "application/json"} if body else {},
        method=method)
    try:
        r = urllib.request.urlopen(req)
        return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()

print("健康 :", hit("GET", "/api/health"))
print("足し算:", hit("POST", "/api/sum", {"a": 1, "b": 2}))
print("型違反:", hit("POST", "/api/sum", {"a": "x", "b": 2}))
print("欠損 :", hit("POST", "/api/sum", {"a": 1}))
print("壊れ :", ("--", "(invalid json で 400 が返る:)"))
import urllib.request as ur
req = ur.Request("http://127.0.0.1:8006/api/sum",
                 data=b"not json", headers={"Content-Type": "application/json"},
                 method="POST")
try: ur.urlopen(req)
except urllib.error.HTTPError as e: print("        ", e.code, e.read().decode())

server.shutdown()


**期待される出力:**
```
健康 : (200, '{"status": "ok"}')
足し算: (200, '{"answer": 3}')
型違反: (400, '{"error": "a と b は数値で"}')
欠損 : (400, '{"error": "a と b が必要"}')
壊れ : (--, '(invalid json で 400 が返る:)')
         400 {"error": "invalid json: Expecting value: line 1 column 1 (char 0)"}
```

🎉 これで HTTP サーバーの足場が完成。
**`exercises/ch01/step6_error_handling.py` と `server/app.py` を見比べて**、
ほぼ同じ骨格になっていることを確認してから次へ。


---
# 第 2 章 — データベース設計

ここからは SQL を打ちます。 Notebook 上で `sqlite3` モジュールを直接使い、
インメモリのデータベースに対して `CREATE TABLE` → `INSERT` → `SELECT` を順番に動かします。

> SQLite と PostgreSQL では一部記法が違いますが、本章で扱う範囲はほぼ同じです。
> `BIGSERIAL` (PG) ↔ `INTEGER PRIMARY KEY AUTOINCREMENT` (SQLite) のような違いだけ後で吸収します。


In [ ]:
import sqlite3

# 1 つの接続を Notebook 全体で使い回す (= ステップが積み上がる)
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")  # SQLiteはデフォOFF。FKを効かせるため必須

def run(sql, params=()):
    """SQL を実行して結果があれば表として表示するヘルパ。"""
    cur = conn.execute(sql, params)
    if cur.description:
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        print(" | ".join(cols))
        print("-+-".join("-" * len(c) for c in cols))
        for row in rows: print(" | ".join(str(v) for v in row))
        print(f"({len(rows)} rows)")
    else:
        print(f"OK ({cur.rowcount} rows affected)")

print("DB connected.")


## Step 1 — `users` テーブルを作って遊ぶ

主キー、UNIQUE、CHECK、DEFAULT、NOT NULL — 制約のオンパレード。


In [ ]:
run('''
CREATE TABLE users (
    id          INTEGER  PRIMARY KEY AUTOINCREMENT,
    name        TEXT     NOT NULL UNIQUE,
    pass_hash   TEXT     NOT NULL,
    coins       INTEGER  NOT NULL DEFAULT 1000 CHECK (coins >= 0),
    created_at  TEXT     NOT NULL DEFAULT (datetime('now'))
)
''')


In [ ]:
run("INSERT INTO users (name, pass_hash) VALUES ('alice', 'fakehash1')")
run("INSERT INTO users (name, pass_hash) VALUES ('bob',   'fakehash2')")
run("INSERT INTO users (name, pass_hash) VALUES ('carol', 'fakehash3')")
run("SELECT id, name, coins, created_at FROM users ORDER BY id")


In [ ]:
# 制約に殴られる体験
try:
    run("INSERT INTO users (name, pass_hash) VALUES ('alice', 'dup')")  # UNIQUE 違反
except Exception as e:
    print("UNIQUE 違反:", e)

try:
    run("UPDATE users SET coins = -1 WHERE name = 'alice'")  # CHECK 違反
except Exception as e:
    print("CHECK 違反:", e)


DB が「自分から間違いを教えてくれる」のが分かったでしょうか。
これがアプリ層に同じバリデーションを書かなくて済む理由です。


## Step 2 — `characters` マスタを足す


In [ ]:
run('''
CREATE TABLE characters (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    name    TEXT    NOT NULL UNIQUE,
    rarity  INTEGER NOT NULL CHECK (rarity BETWEEN 1 AND 5),
    emoji   TEXT    NOT NULL DEFAULT '❓'
)
''')

# データ投入
chars = [
    ('スライム', 1, '🟢'),
    ('ウルフ',   2, '🐺'),
    ('ナイト',   3, '🛡️'),
    ('ペガサス', 4, '🦄'),
    ('ドラゴン', 5, '🐉'),
]
conn.executemany("INSERT INTO characters (name, rarity, emoji) VALUES (?, ?, ?)", chars)

run("SELECT id, name, rarity, emoji FROM characters ORDER BY rarity DESC")


## Step 3 — ガチャ筐体 + 排出設定 (多対多 + 重み)

「複数のガチャに、複数のキャラが、それぞれ違う重みで登場する」を
RDB で表現する常套手段が **中間テーブル**。さらに今回はそこに `weight` (重み) も持たせます。


In [ ]:
run('''
CREATE TABLE gachas (
    id    INTEGER PRIMARY KEY AUTOINCREMENT,
    name  TEXT    NOT NULL UNIQUE,
    price INTEGER NOT NULL CHECK (price > 0)
)
''')

run('''
CREATE TABLE gacha_items (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    gacha_id     INTEGER NOT NULL REFERENCES gachas(id)     ON DELETE CASCADE,
    character_id INTEGER NOT NULL REFERENCES characters(id) ON DELETE RESTRICT,
    weight       INTEGER NOT NULL CHECK (weight > 0),
    UNIQUE (gacha_id, character_id)
)
''')

conn.executemany("INSERT INTO gachas (name, price) VALUES (?, ?)",
                 [('通常ガチャ', 100), ('プレミアムガチャ', 300)])

# 通常ガチャは全レア度をバランス良く
conn.execute('''
INSERT INTO gacha_items (gacha_id, character_id, weight)
SELECT g.id, c.id,
       CASE c.rarity WHEN 1 THEN 60 WHEN 2 THEN 30
                     WHEN 3 THEN 20 WHEN 4 THEN 8 WHEN 5 THEN 2 END
  FROM gachas g, characters c
 WHERE g.name = '通常ガチャ'
''')

# プレミアムは レア度3以上のみ
conn.execute('''
INSERT INTO gacha_items (gacha_id, character_id, weight)
SELECT g.id, c.id,
       CASE c.rarity WHEN 3 THEN 50 WHEN 4 THEN 20 WHEN 5 THEN 10 END
  FROM gachas g, characters c
 WHERE g.name = 'プレミアムガチャ' AND c.rarity >= 3
''')

# 排出表 (確率付き) を見る
run('''
SELECT g.name AS gacha, c.name AS chara, c.rarity, gi.weight,
       round(gi.weight * 100.0 / sum(gi.weight) over (partition by g.id), 2) AS pct
  FROM gacha_items gi
  JOIN gachas g     ON g.id = gi.gacha_id
  JOIN characters c ON c.id = gi.character_id
 ORDER BY g.id, gi.weight DESC
''')


## Step 4 — `user_characters` (= 履歴 / Box) + トランザクション

ガチャを 1 回引く処理は 「coins -100 / 履歴に1行 INSERT」の 2 つの書き換え。
**両方成功 or 両方失敗** にする必要があります。これがトランザクション。


In [ ]:
run('''
CREATE TABLE user_characters (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id      INTEGER NOT NULL REFERENCES users(id)      ON DELETE CASCADE,
    character_id INTEGER NOT NULL REFERENCES characters(id) ON DELETE RESTRICT,
    obtained_at  TEXT    NOT NULL DEFAULT (datetime('now'))
)
''')
run("CREATE INDEX idx_user_characters_user ON user_characters(user_id)")

# alice が通常ガチャを 1 回引いてスライムが当たった、を SQL で再現
conn.execute("BEGIN")
conn.execute("UPDATE users SET coins = coins - 100 WHERE name = 'alice'")
conn.execute('''
INSERT INTO user_characters (user_id, character_id)
SELECT u.id, c.id FROM users u, characters c
 WHERE u.name = 'alice' AND c.name = 'スライム'
''')
conn.execute("COMMIT")

run("SELECT name, coins FROM users WHERE name = 'alice'")
run('''
SELECT u.name AS user, c.name AS chara, uc.obtained_at
  FROM user_characters uc
  JOIN users      u ON u.id = uc.user_id
  JOIN characters c ON c.id = uc.character_id
 ORDER BY uc.id
''')


## Step 5 — JOIN + GROUP BY + ウィンドウ関数で Box を出す

ガチャを何回も引いて、所持品 (Box) を SQL 1 本で集計する練習。


In [ ]:
# alice にあと数回引かせる (ロジックは Step 4 と同じ。今回は色んなキャラに)
import random
random.seed(0)
for chara in ['スライム', 'ウルフ', 'スライム', 'ナイト', 'ウルフ', 'ペガサス', 'スライム']:
    conn.execute("BEGIN")
    conn.execute("UPDATE users SET coins = coins + 100 WHERE name='alice'")  # テスト用 補充
    conn.execute("UPDATE users SET coins = coins - 100 WHERE name='alice'")
    conn.execute('''
        INSERT INTO user_characters (user_id, character_id)
        SELECT u.id, c.id FROM users u, characters c
         WHERE u.name = 'alice' AND c.name = ?
    ''', (chara,))
    conn.execute("COMMIT")

# Box: 所持キャラを個数付きで
print("== alice の Box ==")
run('''
SELECT c.name, c.rarity, c.emoji, COUNT(*) AS count
  FROM user_characters uc
  JOIN characters c ON c.id = uc.character_id
  JOIN users      u ON u.id = uc.user_id
 WHERE u.name = 'alice'
 GROUP BY c.id
 ORDER BY c.rarity DESC, c.id
''')

# 全キャラ + 所持数 (LEFT JOIN: 未所持も含めて出す)
print("\n== 全キャラ × alice の所持状況 ==")
run('''
SELECT c.name, c.rarity, COUNT(uc.id) AS owned
  FROM characters c
  LEFT JOIN user_characters uc
         ON uc.character_id = c.id
        AND uc.user_id = (SELECT id FROM users WHERE name='alice')
 GROUP BY c.id
 ORDER BY c.rarity DESC, c.id
''')


ここまでが第 2 章のキモ。
これで `server/app.py` の `GET /api/box` が発行している SQL がそのまま読めるようになります。


---
# 第 3〜8 章 (圧縮版) — 本物のサーバー一式を書き出す

ここから先 (Postgres / psycopg / 認証 / セッション / ガチャ抽選 / Box / フロント) は、
**完成版コードを `%%writefile` で 一気に書き出して、動かしながら読む** 形で進めます。

各ファイルは「読み物としてもう書いてある」 ので、冒頭の docstring とコメントを
追ってください。コピペした時点でフルセットがそろいます。


## `db/schema.sql` — テーブル定義 (PostgreSQL方言)


In [ ]:
%%writefile db/schema.sql
-- ===========================================================================
-- gacha_gacha スキーマ定義
--
-- 学習用に「正規化された 5 テーブル + セッション 1 テーブル」で構成。
-- 各 CREATE TABLE の前に "なぜこの形なのか" を日本語コメントで書いています。
-- ===========================================================================

-- ---------------------------------------------------------------------------
-- users: ログインアカウント
-- ---------------------------------------------------------------------------
-- name はログインID。display_name は画面表示用。
-- パスワードは平文で持たない。pbkdf2_hmac でハッシュ化したものを格納する。
-- ハッシュアルゴリズム + イテレーション回数 + ソルト + ダイジェストを
-- "$" 区切りで 1 カラムに詰めている (passlib 形式の簡易版)。
-- coins はガチャを回すための仮想通貨 (初期 1000)。
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS users (
    id           BIGSERIAL    PRIMARY KEY,
    name         TEXT         NOT NULL UNIQUE,
    pass_hash    TEXT         NOT NULL,
    display_name TEXT         NOT NULL,
    coins        INTEGER      NOT NULL DEFAULT 1000,
    created_at   TIMESTAMPTZ  NOT NULL DEFAULT NOW()
);

-- ---------------------------------------------------------------------------
-- characters: ガチャから出るキャラ (もしくはアイテム)
-- ---------------------------------------------------------------------------
-- rarity は 1..5 の整数。クライアントで星の数として表示する。
-- emoji は雰囲気づくり。画像URLでも構わない。
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS characters (
    id      BIGSERIAL PRIMARY KEY,
    name    TEXT      NOT NULL UNIQUE,
    rarity  SMALLINT  NOT NULL CHECK (rarity BETWEEN 1 AND 5),
    emoji   TEXT      NOT NULL DEFAULT '❓'
);

-- ---------------------------------------------------------------------------
-- gachas: ガチャ筐体 (どのガチャを引くか)
-- ---------------------------------------------------------------------------
-- 同時に複数の "ガチャ" を運用できるようにテーブル化している。
-- price は 1 回引くのに必要な coins。
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS gachas (
    id    BIGSERIAL PRIMARY KEY,
    name  TEXT      NOT NULL UNIQUE,
    price INTEGER   NOT NULL CHECK (price > 0)
);

-- ---------------------------------------------------------------------------
-- gacha_items: 「どのガチャから、どのキャラが、どのくらいの確率で出るか」
-- ---------------------------------------------------------------------------
-- (gacha_id, character_id) の組み合わせは 1 行。
-- 確率は weight で表す。実確率は SUM(weight) との比で決まる。
-- これにより
--   ・後から weight を変えるだけで確率調整できる
--   ・ガチャごとに排出キャラを完全に分離できる
-- というメリットがある。
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS gacha_items (
    id           BIGSERIAL PRIMARY KEY,
    gacha_id     BIGINT    NOT NULL REFERENCES gachas(id)     ON DELETE CASCADE,
    character_id BIGINT    NOT NULL REFERENCES characters(id) ON DELETE RESTRICT,
    weight       INTEGER   NOT NULL CHECK (weight > 0),
    UNIQUE (gacha_id, character_id)
);

CREATE INDEX IF NOT EXISTS idx_gacha_items_gacha ON gacha_items(gacha_id);

-- ---------------------------------------------------------------------------
-- user_characters: ユーザー所持品 (= Box)
-- ---------------------------------------------------------------------------
-- ガチャを 1 回引くごとに 1 行 INSERT される (重複OK = 同じキャラが何枚でも出る)
-- 「所持しているか」は EXISTS / COUNT で問い合わせる。
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS user_characters (
    id           BIGSERIAL   PRIMARY KEY,
    user_id      BIGINT      NOT NULL REFERENCES users(id)      ON DELETE CASCADE,
    character_id BIGINT      NOT NULL REFERENCES characters(id) ON DELETE RESTRICT,
    obtained_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_user_characters_user ON user_characters(user_id);
CREATE INDEX IF NOT EXISTS idx_user_characters_user_char
    ON user_characters(user_id, character_id);

-- ---------------------------------------------------------------------------
-- sessions: クッキーで持つセッショントークン
-- ---------------------------------------------------------------------------
-- 学習用なので「DB に置く」最も素直な形にしている。
-- token は十分に長いランダム文字列 (256bit base64url) を入れる。
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS sessions (
    token      TEXT        PRIMARY KEY,
    user_id    BIGINT      NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    expires_at TIMESTAMPTZ NOT NULL
);

CREATE INDEX IF NOT EXISTS idx_sessions_user ON sessions(user_id);


## `db/seed.sql` — キャラ/ガチャ/排出設定 の初期データ


In [ ]:
%%writefile db/seed.sql
-- ===========================================================================
-- 初期データ
-- ・キャラクター 12 体 (レア度 1〜5)
-- ・ガチャ筐体 2 個 (通常ガチャ / レアガチャ)
-- ・各ガチャの排出設定
--
-- ON CONFLICT DO NOTHING を付けてあるので、何度実行しても二重投入されない。
-- ===========================================================================

INSERT INTO characters (id, name, rarity, emoji) VALUES
    ( 1, 'スライム',       1, '🟢'),
    ( 2, 'コウモリ',       1, '🦇'),
    ( 3, 'ゴブリン',       1, '👺'),
    ( 4, 'ウルフ',         2, '🐺'),
    ( 5, 'マーメイド',     2, '🧜'),
    ( 6, 'ナイト',         3, '🛡️'),
    ( 7, 'ウィザード',     3, '🧙'),
    ( 8, 'ペガサス',       4, '🦄'),
    ( 9, 'ドラゴンの卵',   4, '🥚'),
    (10, 'ドラゴン',       5, '🐉'),
    (11, '不死鳥',         5, '🔥'),
    (12, '伝説の勇者',     5, '⚔️')
ON CONFLICT (id) DO NOTHING;

-- characters の id は手で振ったので sequence を進める
SELECT setval(pg_get_serial_sequence('characters', 'id'),
              (SELECT MAX(id) FROM characters));

INSERT INTO gachas (id, name, price) VALUES
    (1, '通常ガチャ',      100),
    (2, 'プレミアムガチャ', 300)
ON CONFLICT (id) DO NOTHING;

SELECT setval(pg_get_serial_sequence('gachas', 'id'),
              (SELECT MAX(id) FROM gachas));

-- ---------------------------------------------------------------------------
-- 通常ガチャ: コモン多め、SSR はちょこっと
--   weight 合計 = 100 + 100 + 100 + 60 + 60 + 30 + 30 + 10 + 10 + 2 + 2 + 1 = 505
--   → スライム = 100/505 ≒ 19.8%, 伝説の勇者 = 1/505 ≒ 0.2%
-- ---------------------------------------------------------------------------
INSERT INTO gacha_items (gacha_id, character_id, weight) VALUES
    (1,  1, 100), (1,  2, 100), (1,  3, 100),
    (1,  4,  60), (1,  5,  60),
    (1,  6,  30), (1,  7,  30),
    (1,  8,  10), (1,  9,  10),
    (1, 10,   2), (1, 11,   2), (1, 12,   1)
ON CONFLICT DO NOTHING;

-- ---------------------------------------------------------------------------
-- プレミアムガチャ: レア度3以上のみ。SSRは出やすめ。
-- ---------------------------------------------------------------------------
INSERT INTO gacha_items (gacha_id, character_id, weight) VALUES
    (2,  6,  50), (2,  7,  50),
    (2,  8,  25), (2,  9,  25),
    (2, 10,   8), (2, 11,   8), (2, 12,   4)
ON CONFLICT DO NOTHING;


## `server/__init__.py`


In [ ]:
%%writefile server/__init__.py


## `server/db.py` — DB 接続 (psycopg を使う本番用)

このファイルは PostgreSQL 用です。
あとで Notebook 用の **SQLite 互換アダプタ** をかぶせて切り替えます。


In [ ]:
%%writefile server/db.py
"""DB 接続ヘルパー。

学習用なので、ORM や ConnectionPool 抜きの**最小構成**にしてある。
本番では psycopg_pool.ConnectionPool を使うのが定番だが、
"接続を 1 本張って SQL を投げて受け取る" という核を見せたいので、
ここでは「1 リクエスト = 1 connection」にしている。

`with get_conn() as conn:` で使うと、ブロックを抜けるときに
コミットまたはロールバックしてくれる (psycopg の仕様)。
"""

from __future__ import annotations

import os
from contextlib import contextmanager
from typing import Iterator

import psycopg
from psycopg.rows import dict_row


def _dsn() -> str:
    """環境変数から接続文字列を組み立てる。

    .env 等で設定された値を優先し、無ければ docker-compose のデフォルトを使う。
    """
    host = os.getenv("PGHOST", "localhost")
    port = os.getenv("PGPORT", "5432")
    db = os.getenv("PGDATABASE", "gacha")
    user = os.getenv("PGUSER", "gacha")
    pw = os.getenv("PGPASSWORD", "gacha")
    return f"host={host} port={port} dbname={db} user={user} password={pw}"


@contextmanager
def get_conn() -> Iterator[psycopg.Connection]:
    """1 リクエスト分の DB 接続を提供する。

    使用例:
        with get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT 1")
                row = cur.fetchone()
    """
    # row_factory=dict_row を指定すると、結果が tuple ではなく dict で返る。
    # クライアントへ JSON で返したい今回の用途と相性が良い。
    conn = psycopg.connect(_dsn(), row_factory=dict_row, autocommit=False)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def healthcheck() -> bool:
    """DB に届くか確認する。サーバー起動時の早期失敗用。"""
    try:
        with get_conn() as conn, conn.cursor() as cur:
            cur.execute("SELECT 1")
            cur.fetchone()
        return True
    except Exception as e:
        print(f"[db] healthcheck failed: {e}")
        return False


## `server/auth.py` — パスワードハッシュ + セッション


In [ ]:
%%writefile server/auth.py
"""認証まわり (パスワードハッシュ + セッショントークン)。

学習目的のため、外部依存に頼らず Python 標準ライブラリだけで実装している。

- パスワードハッシュ: hashlib.pbkdf2_hmac (sha256, 200_000 回)
  本番では argon2 や bcrypt を使うのが望ましい。学習用としては
  「ソルトを 1 ユーザーごとに作る」「ハッシュにアルゴリズム情報を埋め込む」
  という考え方を体感するのが目的。

- セッション: 32 byte の暗号学的乱数を base64url にして DB に保存する。
  クライアントには Set-Cookie で返す。
"""

from __future__ import annotations

import base64
import hashlib
import hmac
import os
import secrets
from datetime import datetime, timedelta, timezone
from typing import Optional

import psycopg

# pbkdf2 のパラメータ。学習しやすいよう小さめだが、200_000 はそこそこ実用的。
_PBKDF2_ALGO = "sha256"
_PBKDF2_ITER = 200_000
_PBKDF2_SALT_BYTES = 16
_PBKDF2_DKLEN = 32

# セッションの有効期間
SESSION_TTL = timedelta(days=7)


# ---------------------------------------------------------------------------
# パスワード
# ---------------------------------------------------------------------------
def hash_password(password: str) -> str:
    """パスワードをハッシュ化して、保存用文字列を返す。

    返り値の形式:
        pbkdf2_sha256$<iter>$<salt_b64>$<hash_b64>
    こうしておくとアルゴリズムや反復回数をあとから上げたいときに、
    既存ハッシュと共存しながら段階移行できる。
    """
    salt = os.urandom(_PBKDF2_SALT_BYTES)
    dk = hashlib.pbkdf2_hmac(_PBKDF2_ALGO, password.encode("utf-8"),
                             salt, _PBKDF2_ITER, dklen=_PBKDF2_DKLEN)
    return "$".join([
        f"pbkdf2_{_PBKDF2_ALGO}",
        str(_PBKDF2_ITER),
        base64.b64encode(salt).decode(),
        base64.b64encode(dk).decode(),
    ])


def verify_password(password: str, stored: str) -> bool:
    """保存されたハッシュ文字列と入力パスワードを比較する。"""
    try:
        algo, iter_s, salt_b64, hash_b64 = stored.split("$")
    except ValueError:
        return False
    if not algo.startswith("pbkdf2_"):
        return False
    digest_name = algo.split("_", 1)[1]
    iters = int(iter_s)
    salt = base64.b64decode(salt_b64)
    expected = base64.b64decode(hash_b64)
    actual = hashlib.pbkdf2_hmac(digest_name, password.encode("utf-8"),
                                 salt, iters, dklen=len(expected))
    # タイミング攻撃に強い比較。
    return hmac.compare_digest(actual, expected)


# ---------------------------------------------------------------------------
# セッション
# ---------------------------------------------------------------------------
def new_session_token() -> str:
    """十分に長いランダムトークンを生成する (256bit, URL-safe)。"""
    return secrets.token_urlsafe(32)


def create_session(conn: psycopg.Connection, user_id: int) -> tuple[str, datetime]:
    """sessions テーブルにレコードを差し込み、(token, expires_at) を返す。"""
    token = new_session_token()
    expires_at = datetime.now(timezone.utc) + SESSION_TTL
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO sessions (token, user_id, expires_at) VALUES (%s, %s, %s)",
            (token, user_id, expires_at),
        )
    return token, expires_at


def lookup_session(conn: psycopg.Connection, token: str) -> Optional[dict]:
    """token から user 情報を引く。期限切れは None を返す + 自動削除。"""
    if not token:
        return None
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT u.id, u.name, u.display_name, u.coins, s.expires_at
              FROM sessions s
              JOIN users    u ON u.id = s.user_id
             WHERE s.token = %s
            """,
            (token,),
        )
        row = cur.fetchone()
    if row is None:
        return None
    if row["expires_at"] <= datetime.now(timezone.utc):
        delete_session(conn, token)
        return None
    return {
        "id": row["id"],
        "name": row["name"],
        "display_name": row["display_name"],
        "coins": row["coins"],
    }


def delete_session(conn: psycopg.Connection, token: str) -> None:
    with conn.cursor() as cur:
        cur.execute("DELETE FROM sessions WHERE token = %s", (token,))


## `server/gacha.py` — 重み付き乱択でガチャを 1 回引く


In [ ]:
%%writefile server/gacha.py
"""ガチャ抽選ロジック。

ここがゲームバックエンドで一番面白い部分。
gacha_items テーブルから (character_id, weight) を引いてきて、
weight の合計を分母にした重み付き乱択を行う。

- 単純実装: weight をリストにして bisect/random.choices で 1 個選ぶ
- DB 一発で抽選するパターンも紹介可能 (ORDER BY -log(random())/weight LIMIT 1)
  → 学習として両方のやり方に触れたいので、Pythonでやるバージョンと
    SQLだけでやるバージョンを Ch.6 で比較する。

このモジュールは "Pythonで重み付き乱択" のシンプルな実装。
"""

from __future__ import annotations

import random
from dataclasses import dataclass

import psycopg


@dataclass
class GachaItem:
    character_id: int
    name: str
    rarity: int
    emoji: str
    weight: int


def fetch_pool(conn: psycopg.Connection, gacha_id: int) -> list[GachaItem]:
    """指定ガチャの排出プールを (character情報 + weight) で取ってくる。"""
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT c.id   AS character_id,
                   c.name,
                   c.rarity,
                   c.emoji,
                   gi.weight
              FROM gacha_items gi
              JOIN characters c ON c.id = gi.character_id
             WHERE gi.gacha_id = %s
             ORDER BY c.rarity, c.id
            """,
            (gacha_id,),
        )
        rows = cur.fetchall()
    return [GachaItem(**r) for r in rows]


def draw(pool: list[GachaItem], rng: random.Random | None = None) -> GachaItem:
    """重み付き乱択を 1 回行う。

    実装は random.choices に任せている。
    手で書くなら:

        total = sum(item.weight for item in pool)
        r = rng.uniform(0, total)
        acc = 0
        for item in pool:
            acc += item.weight
            if r <= acc:
                return item

    という累積和の二分探索パターンを書くことになる。Ch.6 で詳しく解説する。
    """
    if not pool:
        raise ValueError("ガチャに排出設定が 1 件もありません")
    rng = rng or random.SystemRandom()  # 暗号学的乱数で
    weights = [item.weight for item in pool]
    return rng.choices(pool, weights=weights, k=1)[0]


## `server/app.py` — http.server + ルーター + 全 API


In [ ]:
%%writefile server/app.py
"""http.server だけで書かれたガチャ API。

なぜフレームワークを使わないのか:
  Flask や FastAPI を使うと「リクエストが来てから JSON を返すまで」の途中が
  全部隠されてしまう。学習目的では、HTTP メソッド + パス + ヘッダ + ボディが
  どう Python のオブジェクトに変換され、何をもって 200 / 400 / 401 を返すのか、
  全部目に見える状態にしておきたい。

エンドポイント:
  POST /api/register
  POST /api/login
  POST /api/logout              (要認証)
  GET  /api/me                  (要認証)
  GET  /api/gacha/list
  POST /api/gacha/pull          (要認証)
  GET  /api/box                 (要認証)

使い方:
  $ python -m server.app
  → http://localhost:8000 で待ち受け開始
"""

from __future__ import annotations

import http.cookies
import json
import os
import sys
import traceback
from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any, Callable
from urllib.parse import urlparse

from . import auth, gacha
from .db import get_conn, healthcheck

# ---------------------------------------------------------------------------
# 設定
# ---------------------------------------------------------------------------
APP_PORT = int(os.getenv("APP_PORT", "8000"))

# CORS: 開発時にフロントを別ポート (例: 5500) で配信できるよう、
# 学習用としてゆるめに開けてある。本番ではきっちり絞る。
ALLOWED_ORIGINS = {
    "http://localhost:5500",
    "http://127.0.0.1:5500",
    f"http://localhost:{APP_PORT}",
    f"http://127.0.0.1:{APP_PORT}",
}

SESSION_COOKIE = "gg_session"


# ---------------------------------------------------------------------------
# 例外クラス
# ---------------------------------------------------------------------------
class AppError(Exception):
    """アプリケーション層で投げる、HTTPステータス付きのエラー。"""

    def __init__(self, status: int, message: str):
        super().__init__(message)
        self.status = status
        self.message = message


def bad_request(msg: str) -> AppError: return AppError(400, msg)
def unauthorized(msg: str = "ログインが必要です") -> AppError: return AppError(401, msg)
def not_found(msg: str = "見つかりません") -> AppError: return AppError(404, msg)


# ---------------------------------------------------------------------------
# ルーター: (method, path) -> handler
# ---------------------------------------------------------------------------
Handler = Callable[["GachaHandler"], Any]
ROUTES: dict[tuple[str, str], Handler] = {}


def route(method: str, path: str):
    def deco(fn: Handler) -> Handler:
        ROUTES[(method.upper(), path)] = fn
        return fn
    return deco


# ---------------------------------------------------------------------------
# リクエストハンドラ
# ---------------------------------------------------------------------------
class GachaHandler(BaseHTTPRequestHandler):
    # 既定のログをカスタマイズ (見やすく)
    def log_message(self, fmt: str, *args: Any) -> None:
        sys.stderr.write(f"[{self.command}] {self.path} -> {fmt % args}\n")

    # ---------- レスポンス補助 ----------
    def _cors_headers(self) -> None:
        origin = self.headers.get("Origin", "")
        if origin in ALLOWED_ORIGINS:
            self.send_header("Access-Control-Allow-Origin", origin)
            self.send_header("Access-Control-Allow-Credentials", "true")
            self.send_header("Vary", "Origin")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")

    def send_json(self, status: int, payload: Any,
                  set_cookie: http.cookies.SimpleCookie | None = None) -> None:
        body = json.dumps(payload, ensure_ascii=False, default=str).encode("utf-8")
        self.send_response(status)
        self._cors_headers()
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        if set_cookie is not None:
            for morsel in set_cookie.values():
                self.send_header("Set-Cookie", morsel.OutputString())
        self.end_headers()
        self.wfile.write(body)

    # ---------- リクエスト解析 ----------
    def read_json(self) -> dict:
        length = int(self.headers.get("Content-Length", "0"))
        if length == 0:
            return {}
        raw = self.rfile.read(length)
        try:
            data = json.loads(raw.decode("utf-8"))
        except json.JSONDecodeError as e:
            raise bad_request(f"JSON が読めません: {e}")
        if not isinstance(data, dict):
            raise bad_request("JSON の最上位はオブジェクトにしてください")
        return data

    def get_cookie(self, name: str) -> str:
        raw = self.headers.get("Cookie", "")
        if not raw:
            return ""
        jar = http.cookies.SimpleCookie()
        jar.load(raw)
        morsel = jar.get(name)
        return morsel.value if morsel else ""

    def require_user(self, conn) -> dict:
        token = self.get_cookie(SESSION_COOKIE)
        user = auth.lookup_session(conn, token)
        if user is None:
            raise unauthorized()
        return user

    # ---------- ディスパッチ ----------
    def do_OPTIONS(self) -> None:  # noqa: N802 - http.server convention
        self.send_response(HTTPStatus.NO_CONTENT)
        self._cors_headers()
        self.send_header("Content-Length", "0")
        self.end_headers()

    def do_GET(self) -> None:  # noqa: N802
        self._dispatch("GET")

    def do_POST(self) -> None:  # noqa: N802
        self._dispatch("POST")

    def _dispatch(self, method: str) -> None:
        path = urlparse(self.path).path
        handler = ROUTES.get((method, path))
        if handler is None:
            self.send_json(404, {"error": "Not Found", "path": path})
            return
        try:
            handler(self)
        except AppError as e:
            self.send_json(e.status, {"error": e.message})
        except Exception as e:  # noqa: BLE001
            traceback.print_exc()
            self.send_json(500, {"error": "internal server error", "detail": str(e)})


# ---------------------------------------------------------------------------
# 共通: ユーザー情報を JSON 用に整える
# ---------------------------------------------------------------------------
def _user_payload(u: dict) -> dict:
    return {
        "id": u["id"],
        "name": u["name"],
        "display_name": u["display_name"],
        "coins": u["coins"],
    }


def _make_session_cookie(token: str) -> http.cookies.SimpleCookie:
    jar = http.cookies.SimpleCookie()
    jar[SESSION_COOKIE] = token
    m = jar[SESSION_COOKIE]
    m["path"] = "/"
    m["httponly"] = True
    m["samesite"] = "Lax"
    m["max-age"] = int(auth.SESSION_TTL.total_seconds())
    return jar


def _clear_session_cookie() -> http.cookies.SimpleCookie:
    jar = http.cookies.SimpleCookie()
    jar[SESSION_COOKIE] = ""
    m = jar[SESSION_COOKIE]
    m["path"] = "/"
    m["httponly"] = True
    m["samesite"] = "Lax"
    m["max-age"] = 0
    return jar


# ===========================================================================
# ルート定義
# ===========================================================================
@route("POST", "/api/register")
def register(h: GachaHandler) -> None:
    body = h.read_json()
    name = (body.get("name") or "").strip()
    password = body.get("password") or ""
    display_name = (body.get("display_name") or "").strip() or name
    if len(name) < 3 or len(name) > 32:
        raise bad_request("name は 3〜32 文字")
    if len(password) < 6:
        raise bad_request("password は 6 文字以上")

    pass_hash = auth.hash_password(password)
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1 FROM users WHERE name = %s", (name,))
            if cur.fetchone():
                raise bad_request("その name は既に使われています")
            cur.execute(
                """
                INSERT INTO users (name, pass_hash, display_name)
                VALUES (%s, %s, %s)
                RETURNING id, name, display_name, coins
                """,
                (name, pass_hash, display_name),
            )
            user = cur.fetchone()
        token, _ = auth.create_session(conn, user["id"])

    h.send_json(200, {"user": _user_payload(user)},
                set_cookie=_make_session_cookie(token))


@route("POST", "/api/login")
def login(h: GachaHandler) -> None:
    body = h.read_json()
    name = (body.get("name") or "").strip()
    password = body.get("password") or ""
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT id, name, pass_hash, display_name, coins "
                "FROM users WHERE name = %s",
                (name,),
            )
            user = cur.fetchone()
        if user is None or not auth.verify_password(password, user["pass_hash"]):
            raise AppError(401, "name か password が違います")
        token, _ = auth.create_session(conn, user["id"])

    h.send_json(200, {"user": _user_payload(user)},
                set_cookie=_make_session_cookie(token))


@route("POST", "/api/logout")
def logout(h: GachaHandler) -> None:
    token = h.get_cookie(SESSION_COOKIE)
    if token:
        with get_conn() as conn:
            auth.delete_session(conn, token)
    h.send_json(200, {"ok": True}, set_cookie=_clear_session_cookie())


@route("GET", "/api/me")
def me(h: GachaHandler) -> None:
    with get_conn() as conn:
        user = h.require_user(conn)
    h.send_json(200, {"user": user})


@route("GET", "/api/gacha/list")
def gacha_list(h: GachaHandler) -> None:
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(
            """
            SELECT g.id, g.name, g.price,
                   COUNT(gi.id)        AS pool_size,
                   COALESCE(SUM(gi.weight), 0) AS total_weight
              FROM gachas g
              LEFT JOIN gacha_items gi ON gi.gacha_id = g.id
             GROUP BY g.id
             ORDER BY g.id
            """
        )
        rows = cur.fetchall()
    h.send_json(200, {"gachas": rows})


@route("POST", "/api/gacha/pull")
def gacha_pull(h: GachaHandler) -> None:
    body = h.read_json()
    gacha_id = body.get("gacha_id")
    if not isinstance(gacha_id, int):
        raise bad_request("gacha_id は整数で指定してください")

    with get_conn() as conn:
        user = h.require_user(conn)

        # 1. ガチャを取得 + 価格チェック (FOR UPDATE で行ロック)
        with conn.cursor() as cur:
            cur.execute(
                "SELECT id, name, price FROM gachas WHERE id = %s FOR SHARE",
                (gacha_id,),
            )
            g = cur.fetchone()
            if g is None:
                raise not_found("そのガチャは存在しません")

            cur.execute(
                "SELECT coins FROM users WHERE id = %s FOR UPDATE",
                (user["id"],),
            )
            row = cur.fetchone()
            coins = row["coins"]
            if coins < g["price"]:
                raise bad_request(
                    f"コインが足りません (必要 {g['price']} / 所持 {coins})"
                )

        # 2. 抽選
        pool = gacha.fetch_pool(conn, gacha_id)
        if not pool:
            raise bad_request("そのガチャには排出設定がありません")
        won = gacha.draw(pool)

        # 3. coins を引き、user_characters に INSERT
        with conn.cursor() as cur:
            cur.execute(
                "UPDATE users SET coins = coins - %s WHERE id = %s "
                "RETURNING coins",
                (g["price"], user["id"]),
            )
            new_coins = cur.fetchone()["coins"]

            cur.execute(
                "INSERT INTO user_characters (user_id, character_id) "
                "VALUES (%s, %s) RETURNING obtained_at",
                (user["id"], won.character_id),
            )
            obtained_at = cur.fetchone()["obtained_at"]

    h.send_json(200, {
        "gacha": {"id": g["id"], "name": g["name"], "price": g["price"]},
        "character": {
            "id": won.character_id,
            "name": won.name,
            "rarity": won.rarity,
            "emoji": won.emoji,
        },
        "obtained_at": obtained_at,
        "coins": new_coins,
    })


@route("GET", "/api/box")
def box(h: GachaHandler) -> None:
    with get_conn() as conn:
        user = h.require_user(conn)
        with conn.cursor() as cur:
            # 同じキャラを複数回引けるので、character_id ごとに集計する
            cur.execute(
                """
                SELECT c.id, c.name, c.rarity, c.emoji,
                       COUNT(*)         AS count,
                       MIN(uc.obtained_at) AS first_obtained_at,
                       MAX(uc.obtained_at) AS last_obtained_at
                  FROM user_characters uc
                  JOIN characters c ON c.id = uc.character_id
                 WHERE uc.user_id = %s
                 GROUP BY c.id
                 ORDER BY c.rarity DESC, c.id
                """,
                (user["id"],),
            )
            items = cur.fetchall()

            cur.execute(
                "SELECT COUNT(*) AS total_pulls FROM user_characters "
                "WHERE user_id = %s",
                (user["id"],),
            )
            total_pulls = cur.fetchone()["total_pulls"]

    h.send_json(200, {
        "user": user,
        "total_pulls": total_pulls,
        "items": items,
    })


# ---------------------------------------------------------------------------
# エントリポイント
# ---------------------------------------------------------------------------
def main() -> int:
    if not healthcheck():
        print("PostgreSQL に接続できません。`docker compose up -d` を先に。")
        return 1
    addr = ("0.0.0.0", APP_PORT)
    server = ThreadingHTTPServer(addr, GachaHandler)
    print(f"[gacha_gacha] listening on http://localhost:{APP_PORT}")
    print("  ルート一覧:")
    for (m, p) in sorted(ROUTES):
        print(f"    {m:5s} {p}")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nbye")
    finally:
        server.server_close()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `web/index.html` — ガチャ画面 (HTML)


In [ ]:
%%writefile web/index.html
<!doctype html>
<html lang="ja">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>gacha_gacha</title>
<link rel="stylesheet" href="./style.css">
</head>
<body>
<header>
  <h1>🎰 gacha_gacha</h1>
  <div id="user-area">
    <span id="welcome" hidden></span>
    <button id="logout-btn" hidden>ログアウト</button>
  </div>
</header>

<main>
  <!-- ログイン/登録 -->
  <section id="auth-section">
    <div class="card">
      <h2>ログイン or 新規登録</h2>
      <p class="hint">同じフォームで両方できます。新規ユーザーには 1000 coin が配られます。</p>
      <form id="auth-form">
        <label>name <input name="name" autocomplete="username" required minlength="3"></label>
        <label>password <input name="password" type="password" autocomplete="current-password" required minlength="6"></label>
        <label>display name (登録時のみ) <input name="display_name" autocomplete="nickname"></label>
        <div class="row">
          <button type="submit" data-mode="login">ログイン</button>
          <button type="submit" data-mode="register">新規登録</button>
        </div>
      </form>
      <p id="auth-error" class="error" hidden></p>
    </div>
  </section>

  <!-- ガチャ -->
  <section id="play-section" hidden>
    <div class="card">
      <h2>ガチャを引く</h2>
      <p>所持コイン: <strong id="coins">-</strong></p>
      <div id="gacha-list" class="gacha-list"></div>
      <div id="pull-result" class="pull-result" hidden></div>
    </div>

    <div class="card">
      <h2>📦 Box (所持キャラ)</h2>
      <p><span id="total-pulls">0</span> 回引いた / <span id="unique-count">0</span> 種類</p>
      <button id="reload-box-btn">最新化</button>
      <div id="box" class="box-grid"></div>
    </div>
  </section>
</main>

<footer>
  <small>API: <code id="api-base"></code> ｜ ソース: <code>server/app.py</code></small>
</footer>

<script src="./app.js" type="module"></script>
</body>
</html>


## `web/style.css`


In [ ]:
%%writefile web/style.css
:root {
  --bg: #f7f6f2;
  --card: #ffffff;
  --ink: #1f2937;
  --muted: #6b7280;
  --accent: #4f46e5;
  --accent-ink: #ffffff;
  --danger: #b91c1c;
  --r1: #9ca3af;
  --r2: #34d399;
  --r3: #60a5fa;
  --r4: #c084fc;
  --r5: #f59e0b;
  --shadow: 0 6px 20px rgba(0,0,0,0.06);
}
* { box-sizing: border-box; }
html, body { margin: 0; padding: 0; background: var(--bg); color: var(--ink);
             font-family: -apple-system, BlinkMacSystemFont, "Segoe UI",
                          "Hiragino Sans", "Yu Gothic", sans-serif; }
header { display: flex; justify-content: space-between; align-items: center;
         padding: 16px 24px; background: var(--card); box-shadow: var(--shadow); }
header h1 { margin: 0; font-size: 20px; }
main { max-width: 880px; margin: 24px auto; padding: 0 16px; }
footer { text-align: center; color: var(--muted); padding: 24px 0; }

.card { background: var(--card); padding: 20px 24px; border-radius: 12px;
        box-shadow: var(--shadow); margin-bottom: 20px; }
.card h2 { margin-top: 0; font-size: 18px; }
.hint { color: var(--muted); font-size: 13px; }

form label { display: block; margin: 8px 0; font-size: 14px; color: var(--muted); }
form input { display: block; width: 100%; padding: 8px 10px; margin-top: 4px;
             border: 1px solid #d1d5db; border-radius: 6px; font-size: 14px; }
.row { display: flex; gap: 8px; margin-top: 12px; }
button { background: var(--accent); color: var(--accent-ink); border: none;
         padding: 8px 16px; border-radius: 6px; cursor: pointer;
         font-size: 14px; }
button:hover { filter: brightness(1.05); }
button.secondary { background: #e5e7eb; color: var(--ink); }
button:disabled { opacity: 0.5; cursor: not-allowed; }

.error { color: var(--danger); font-size: 13px; margin-top: 8px; }

.gacha-list { display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr));
              gap: 12px; margin-top: 12px; }
.gacha-card { border: 1px solid #e5e7eb; border-radius: 10px; padding: 14px; }
.gacha-card h3 { margin: 0 0 4px 0; }
.gacha-card .meta { color: var(--muted); font-size: 12px; }

.pull-result { margin-top: 16px; padding: 16px; border-radius: 10px;
               background: linear-gradient(135deg, #fff7ed, #fef3c7); }
.pull-result.rarity-5 { background: linear-gradient(135deg, #fde68a, #fbbf24); }
.pull-result.rarity-4 { background: linear-gradient(135deg, #ede9fe, #c4b5fd); }
.pull-result.rarity-3 { background: linear-gradient(135deg, #dbeafe, #93c5fd); }
.pull-result.rarity-2 { background: linear-gradient(135deg, #d1fae5, #6ee7b7); }
.pull-result.rarity-1 { background: linear-gradient(135deg, #f3f4f6, #d1d5db); }
.pull-result .emoji { font-size: 56px; }
.pull-result .name { font-weight: bold; font-size: 18px; }
.pull-result .stars { letter-spacing: 2px; }

.box-grid { display: grid;
            grid-template-columns: repeat(auto-fit, minmax(110px, 1fr));
            gap: 10px; margin-top: 12px; }
.char { border: 1px solid #e5e7eb; border-radius: 10px; padding: 10px;
        text-align: center; background: #fafafa; position: relative; }
.char .emoji { font-size: 36px; }
.char .name { font-size: 12px; }
.char .count { position: absolute; top: 4px; right: 6px; font-size: 11px;
               background: var(--accent); color: white; border-radius: 999px;
               padding: 2px 6px; }
.char[data-rarity="5"] { border-color: var(--r5); box-shadow: 0 0 0 2px #fde68a; }
.char[data-rarity="4"] { border-color: var(--r4); }
.char[data-rarity="3"] { border-color: var(--r3); }
.char[data-rarity="2"] { border-color: var(--r2); }

#user-area { display: flex; gap: 8px; align-items: center; }


## `web/app.js` — ブラウザ側ロジック


In [ ]:
%%writefile web/app.js
// gacha_gacha フロントエンド (フレームワーク無し / モジュール 1 つ)
// ----------------------------------------------------------------
// 教材的な狙い: fetch で JSON を投げる、Cookie が credentials で送られる、
// 受け取った JSON を DOM に貼る、というブラウザ側の最小ループだけを書く。

const API_BASE = "http://localhost:8000";
document.getElementById("api-base").textContent = API_BASE;

// --- API 呼び出しの薄いラッパ ----------------------------------
async function api(path, { method = "GET", body } = {}) {
  const res = await fetch(API_BASE + path, {
    method,
    credentials: "include",   // ← Cookie をクロスオリジンでも送る
    headers: body ? { "Content-Type": "application/json" } : {},
    body: body ? JSON.stringify(body) : undefined,
  });
  let data = null;
  try { data = await res.json(); } catch (_) { /* 空ボディもOK */ }
  if (!res.ok) {
    const err = new Error(data?.error || `HTTP ${res.status}`);
    err.status = res.status;
    err.payload = data;
    throw err;
  }
  return data;
}

// --- 状態 ------------------------------------------------------
let currentUser = null;

// --- 起動時 ----------------------------------------------------
async function bootstrap() {
  try {
    const { user } = await api("/api/me");
    setUser(user);
    await loadAfterLogin();
  } catch (e) {
    if (e.status === 401) {
      // 未ログイン状態。ログイン画面を出すだけ。
      showAuth();
    } else {
      showError(`接続失敗: ${e.message}`);
    }
  }
}

function setUser(user) {
  currentUser = user;
  document.getElementById("welcome").hidden = false;
  document.getElementById("welcome").textContent =
    `ようこそ ${user.display_name} さん`;
  document.getElementById("logout-btn").hidden = false;
  document.getElementById("auth-section").hidden = true;
  document.getElementById("play-section").hidden = false;
  document.getElementById("coins").textContent = user.coins;
}

function showAuth() {
  document.getElementById("auth-section").hidden = false;
  document.getElementById("play-section").hidden = true;
  document.getElementById("welcome").hidden = true;
  document.getElementById("logout-btn").hidden = true;
}

function showError(msg) {
  const el = document.getElementById("auth-error");
  el.textContent = msg;
  el.hidden = false;
}
function hideError() {
  document.getElementById("auth-error").hidden = true;
}

// --- 認証フォーム ---------------------------------------------
document.getElementById("auth-form").addEventListener("submit", async (ev) => {
  ev.preventDefault();
  hideError();
  const mode = ev.submitter.dataset.mode;   // 'login' or 'register'
  const fd = new FormData(ev.target);
  const payload = {
    name: fd.get("name"),
    password: fd.get("password"),
  };
  if (mode === "register") {
    payload.display_name = fd.get("display_name") || fd.get("name");
  }
  try {
    const { user } = await api(`/api/${mode}`, { method: "POST", body: payload });
    setUser(user);
    await loadAfterLogin();
  } catch (e) {
    showError(e.message);
  }
});

document.getElementById("logout-btn").addEventListener("click", async () => {
  await api("/api/logout", { method: "POST" });
  currentUser = null;
  showAuth();
});

// --- ログイン後ロード ----------------------------------------
async function loadAfterLogin() {
  await Promise.all([loadGachas(), loadBox()]);
}

async function loadGachas() {
  const { gachas } = await api("/api/gacha/list");
  const el = document.getElementById("gacha-list");
  el.innerHTML = "";
  for (const g of gachas) {
    const card = document.createElement("div");
    card.className = "gacha-card";
    card.innerHTML = `
      <h3>${g.name}</h3>
      <div class="meta">価格: ${g.price} coin / 排出 ${g.pool_size} 種</div>
      <div class="row">
        <button data-id="${g.id}">引く</button>
      </div>
    `;
    card.querySelector("button").addEventListener("click", () => pull(g.id));
    el.appendChild(card);
  }
}

async function pull(gacha_id) {
  try {
    const r = await api("/api/gacha/pull", {
      method: "POST", body: { gacha_id },
    });
    document.getElementById("coins").textContent = r.coins;
    showPullResult(r);
    await loadBox();
  } catch (e) {
    showError(e.message);
  }
}

function showPullResult(r) {
  const el = document.getElementById("pull-result");
  el.hidden = false;
  el.className = `pull-result rarity-${r.character.rarity}`;
  el.innerHTML = `
    <div class="emoji">${r.character.emoji}</div>
    <div class="name">${r.character.name}</div>
    <div class="stars">${"★".repeat(r.character.rarity)}${"☆".repeat(5 - r.character.rarity)}</div>
    <div class="meta">${r.gacha.name} で獲得 (-${r.gacha.price} coin)</div>
  `;
}

// --- Box -----------------------------------------------------
document.getElementById("reload-box-btn").addEventListener("click", loadBox);

async function loadBox() {
  const { items, total_pulls } = await api("/api/box");
  document.getElementById("total-pulls").textContent = total_pulls;
  document.getElementById("unique-count").textContent = items.length;
  const el = document.getElementById("box");
  el.innerHTML = "";
  if (items.length === 0) {
    el.innerHTML = `<p class="hint">まだ何も持っていません。ガチャを引いてみよう！</p>`;
    return;
  }
  for (const it of items) {
    const div = document.createElement("div");
    div.className = "char";
    div.dataset.rarity = it.rarity;
    div.innerHTML = `
      <span class="count">×${it.count}</span>
      <div class="emoji">${it.emoji}</div>
      <div class="name">${it.name}</div>
      <div class="stars">${"★".repeat(it.rarity)}</div>
    `;
    el.appendChild(div);
  }
}

bootstrap();


---
# 第 X 章 — Notebook で動かす: SQLite 互換アダプタ

`server/db.py` は psycopg + PostgreSQL 用に書かれているので、
このまま Colab で動かすには Postgres を別途立てる必要があります。

そこで **本番コードに一切手を加えずに**、`server.db.get_conn` だけ
SQLite にしゃべる関数に差し替えるアダプタを用意します。

> 後で本物の Postgres に切り替えたいときは、この差し替えセルを実行**しなければ**
> 元の psycopg ベースに戻ります。 (= 抽象化はここに集約されています)


In [ ]:
%%writefile sqlite_adapter.py
"""Test-only adapter that lets `server/*` run against SQLite.

Production uses psycopg + PostgreSQL. For sandbox testing we substitute a
SQLite-backed connection that mimics the psycopg cursor / connection API the
server code touches:

    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT ... WHERE x = %s", (1,))
            cur.fetchone()  # -> dict
            cur.fetchall()  # -> list[dict]
        # implicit commit on success / rollback on exception

Translation layer:
- `%s` placeholders         -> `?`
- `NOW()`                   -> `datetime('now')`
- `FOR UPDATE` / `FOR SHARE` -> stripped (sqlite is single-writer)
- datetime parameters       -> `'YYYY-MM-DD HH:MM:SS'` UTC string
- `RETURNING` is native in modern SQLite (>=3.35)
"""

from __future__ import annotations

import datetime as _dt
import re
import sqlite3
from contextlib import contextmanager
from typing import Any, Iterator


# ---------------------------------------------------------------------------
# Type conversion: make SQLite's TIMESTAMP columns come back as
# timezone-aware datetime (UTC), to match psycopg's TIMESTAMPTZ behaviour.
# ---------------------------------------------------------------------------
def _convert_ts(raw: bytes) -> _dt.datetime:
    s = raw.decode("ascii")
    # Accept "YYYY-MM-DD HH:MM:SS" and "YYYY-MM-DD HH:MM:SS.ffffff"
    fmt = "%Y-%m-%d %H:%M:%S.%f" if "." in s else "%Y-%m-%d %H:%M:%S"
    return _dt.datetime.strptime(s, fmt).replace(tzinfo=_dt.timezone.utc)


sqlite3.register_converter("TIMESTAMP", _convert_ts)


_FOR_LOCK_RE = re.compile(r"\s+FOR\s+(UPDATE|SHARE)\b", re.IGNORECASE)
_NOW_RE = re.compile(r"\bNOW\(\)", re.IGNORECASE)
_PCT_S_RE = re.compile(r"%s")


def _translate_sql(sql: str) -> str:
    sql = _FOR_LOCK_RE.sub("", sql)
    sql = _NOW_RE.sub("datetime('now')", sql)
    sql = _PCT_S_RE.sub("?", sql)
    return sql


def _translate_param(p: Any) -> Any:
    if isinstance(p, _dt.datetime):
        if p.tzinfo is not None:
            p = p.astimezone(_dt.timezone.utc).replace(tzinfo=None)
        return p.strftime("%Y-%m-%d %H:%M:%S")
    return p


def _translate_params(params: Any) -> Any:
    if params is None:
        return params
    if isinstance(params, dict):
        return {k: _translate_param(v) for k, v in params.items()}
    return tuple(_translate_param(p) for p in params)


class _Cursor:
    def __init__(self, conn: sqlite3.Connection):
        self._cur = conn.cursor()
        self._cur.row_factory = sqlite3.Row

    # cursor protocol used by server code -----------------------------------
    def execute(self, sql: str, params: Any = ()) -> "_Cursor":
        self._cur.execute(_translate_sql(sql), _translate_params(params))
        return self

    def fetchone(self) -> dict | None:
        row = self._cur.fetchone()
        return dict(row) if row is not None else None

    def fetchall(self) -> list[dict]:
        return [dict(r) for r in self._cur.fetchall()]

    def close(self) -> None:
        self._cur.close()

    # context manager
    def __enter__(self): return self
    def __exit__(self, *exc): self.close(); return False


class _Connection:
    """Thin wrapper around sqlite3.Connection that mimics psycopg.Connection."""

    def __init__(self, sqlite_conn: sqlite3.Connection):
        self._conn = sqlite_conn

    def cursor(self) -> _Cursor:
        return _Cursor(self._conn)

    def commit(self) -> None:
        self._conn.commit()

    def rollback(self) -> None:
        self._conn.rollback()

    def close(self) -> None:
        # don't actually close - shared in-memory db across test
        pass


# A single shared in-memory database for the whole test run.
# `:memory:` per connection would be a different DB, so we use a named
# shared cache.
_SHARED_URI = "file:gacha_test?mode=memory&cache=shared"
_keepalive: sqlite3.Connection | None = None


def init_shared_db(schema_sql: str, seed_sql: str) -> None:
    """Initialise the shared in-memory DB. Idempotent (drops then recreates)."""
    global _keepalive
    if _keepalive is not None:
        _keepalive.close()
    _keepalive = sqlite3.connect(_SHARED_URI, uri=True,
                                  detect_types=sqlite3.PARSE_DECLTYPES,
                                  isolation_level=None)  # autocommit
    _keepalive.execute("PRAGMA foreign_keys = ON")
    _keepalive.executescript(schema_sql)
    _keepalive.executescript(seed_sql)


@contextmanager
def get_conn() -> Iterator[_Connection]:
    """Drop-in replacement for `server.db.get_conn` during tests."""
    raw = sqlite3.connect(_SHARED_URI, uri=True,
                           detect_types=sqlite3.PARSE_DECLTYPES,
                           isolation_level="DEFERRED")
    raw.execute("PRAGMA foreign_keys = ON")
    conn = _Connection(raw)
    try:
        yield conn
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()


def healthcheck() -> bool:
    try:
        with get_conn() as c, c.cursor() as cur:
            cur.execute("SELECT 1 AS ok")
            return cur.fetchone()["ok"] == 1
    except Exception:
        return False


In [ ]:
# 本物の DDL を SQLite 用に変換する小ヘルパ
import re

def translate_pg_to_sqlite(sql: str) -> str:
    sql = re.sub(r"SELECT setval\([^;]*?\);", "", sql, flags=re.DOTALL | re.I)
    sql = re.sub(r"\bBIGSERIAL\s+PRIMARY KEY\b",
                 "INTEGER PRIMARY KEY AUTOINCREMENT", sql, flags=re.I)
    sql = re.sub(r"\bBIGSERIAL\b", "INTEGER PRIMARY KEY AUTOINCREMENT", sql, flags=re.I)
    sql = re.sub(r"\bBIGINT\b",   "INTEGER", sql, flags=re.I)
    sql = re.sub(r"\bSMALLINT\b", "INTEGER", sql, flags=re.I)
    sql = re.sub(r"\bTIMESTAMPTZ\b", "TIMESTAMP", sql, flags=re.I)
    sql = re.sub(r"\bDEFAULT NOW\(\)", "DEFAULT (datetime('now'))", sql, flags=re.I)
    return sql

schema_pg   = open("db/schema.sql", encoding="utf-8").read()
seed_pg     = open("db/seed.sql",   encoding="utf-8").read()
schema_lite = translate_pg_to_sqlite(schema_pg)
seed_lite   = translate_pg_to_sqlite(seed_pg)
print(schema_lite[:300], "...")


In [ ]:
# server.db を SQLite 版に差し替える
import importlib, sqlite_adapter, server.db

sqlite_adapter.init_shared_db(schema_lite, seed_lite)
server.db.get_conn = sqlite_adapter.get_conn
server.db.healthcheck = sqlite_adapter.healthcheck

# server.app は import 時に from .db import get_conn しているので、再 import する
import server.app
importlib.reload(server.app)
server.app.get_conn = sqlite_adapter.get_conn

# 接続できるか確認
print("healthcheck:", server.db.healthcheck())


---
# サーバーを起動 (バックグラウンドスレッド)

`server/app.py` の `GachaHandler` を、Notebook プロセス内のスレッドで起動します。
ポート 8000 で待ち受け、Colab なら `serve_kernel_port_as_public` で外向きに公開します。


In [ ]:
from http.server import ThreadingHTTPServer
import threading, time

PORT = 8000
_httpd = ThreadingHTTPServer(("0.0.0.0", PORT), server.app.GachaHandler)
_thread = threading.Thread(target=_httpd.serve_forever, daemon=True)
_thread.start()
time.sleep(0.5)
print(f"server listening on :{PORT}")

# Colab なら公開URLを取得 (外向きにする)
try:
    from google.colab import output as _colab_output
    PUBLIC_URL = _colab_output.serve_kernel_port_as_public(PORT)
    if PUBLIC_URL.endswith("/"):
        PUBLIC_URL = PUBLIC_URL[:-1]
    print("public URL:", PUBLIC_URL)
except Exception:
    PUBLIC_URL = f"http://127.0.0.1:{PORT}"
    print("local URL:", PUBLIC_URL)


---
# API を叩いてみる

`requests` を使って (Colab に標準で入っています)、ユーザー登録 → ガチャ → Box 確認 まで通します。
出力を見ながら、 「これは `server/app.py` のどのハンドラに対応しているのか?」 を
照らし合わせながら進めると吸収が速いです。


In [ ]:
import requests, json, random

s = requests.Session()
USER = f"colab_user_{random.randint(1000, 9999)}"
PWD  = "secret123"

# 1. 登録
r = s.post(PUBLIC_URL + "/api/register",
           json={"name": USER, "password": PWD, "display_name": "コラボ太郎"})
print("register:", r.status_code, r.json())

# 2. /api/me で自分の情報を確認
print("me      :", s.get(PUBLIC_URL + "/api/me").json())

# 3. ガチャ一覧
print("gachas  :", s.get(PUBLIC_URL + "/api/gacha/list").json())

# 4. 通常ガチャを 5 連
print("\n=== 5連ガチャ ===")
for i in range(5):
    r = s.post(PUBLIC_URL + "/api/gacha/pull", json={"gacha_id": 1}).json()
    c = r["character"]
    stars = "★" * c["rarity"] + "☆" * (5 - c["rarity"])
    print(f"  {i+1}. {c['emoji']} {c['name']:8s} {stars}  残コイン: {r['coins']}")

# 5. Box (集計済み)
print("\n=== Box ===")
box = s.get(PUBLIC_URL + "/api/box").json()
for it in box["items"]:
    stars = "★" * it["rarity"]
    print(f"  ×{it['count']:2d}  {it['emoji']}  {it['name']:8s}  {stars}")
print(f"通算 {box['total_pulls']} 回, {len(box['items'])} 種類")


---
# Notebook 内でガチャ画面を表示する

`web/index.html` + `web/style.css` + `web/app.js` を 1 枚の HTML に組み立てて、
`IPython.display.HTML` で表示します。
**API のベース URL を Colab 用の公開 URL に書き換える** のがミソ。


In [ ]:
from IPython.display import HTML, display

html_src = open("web/index.html", encoding="utf-8").read()
css_src  = open("web/style.css", encoding="utf-8").read()
js_src   = open("web/app.js",    encoding="utf-8").read()

# index.html は <link rel="stylesheet" href="./style.css"> と <script src="./app.js" type="module">
# を使っているが、Notebook 内では取れないのでインライン化する
html_src = html_src.replace(
    '<link rel="stylesheet" href="./style.css">',
    f'<style>{css_src}</style>')
html_src = html_src.replace(
    '<script src="./app.js" type="module"></script>',
    f'<script type="module">{js_src}</script>')

# API_BASE をこのセッションの公開 URL に差し替える
html_src = html_src.replace(
    'const API_BASE = "http://localhost:8000";',
    f'const API_BASE = "{PUBLIC_URL}";')

display(HTML(html_src))


**🎉 上のセルの直下に、ガチャ画面が出現します。**

操作の流れ:
1. 上のセクションで作ったユーザー (例: `colab_user_1234` / `secret123`) でログイン、または別途登録
2. 「引く」ボタンでガチャを引く。レア度に応じて演出が変わる
3. 下の「Box」に獲得済みキャラが個数付きで蓄積される

> Notebook の中で動いているので、**サーバーを止めるとこの画面も止まります**。
> 上の方のセルの `_httpd` を `_httpd.shutdown()` で停止できます。


---
# ここから先 — 拡張章の道しるべ

ここまでで「動くガチャシステム」が手元に揃いました。
`server/`, `db/`, `web/` の中身は本物のフルセット。 リポジトリ直下のものと同じです。

## 第 3 章 (本格版) — PostgreSQL を Docker で動かす

Notebook で SQLite を使ってきましたが、本番想定は PostgreSQL です。
**コードは一切変えずに**、リポジトリの `docker-compose.yml` を起動して
`server.db.get_conn` の差し替えセルを **実行しないだけ** で本物の PG に切り替わります。

ローカルで:
```bash
docker compose up -d                                # PG が立ち上がる
pip install -r server/requirements.txt              # psycopg 入れる
python -m server.app                                # 本物起動
python -m http.server 5500 --directory web         # フロント配信
```

## 第 4 章 — psycopg

Notebook が使った `sqlite_adapter` と本物の `server/db.py` を見比べてください。
カーソル → 行 → コミット という流れは同じ。違いは「TCP 越しに PG にしゃべる」ことだけ。

## 第 5〜8 章 — 認証 / 抽選 / Box / セッション

それぞれ `server/auth.py`, `server/gacha.py`, `server/app.py` の各ハンドラを
ゆっくり読み直すのが宿題です。 docstring とインラインコメントが「本文」 になっています。

## 第 9 章 — 拡張: クッキークリッカー化

ガチャを軸にゲームを膨らませる方向の話 (ログインボーナス、装備、レベル、リアルタイム…)。
リポジトリの `docs/ch09-extending.html` に骨子があります。

## テスト

```bash
python tests/test_unit.py        # auth + gacha のユニットテスト
python tests/test_e2e_sqlite.py  # SQLite で API を E2E 検証 (27 ケース)
bash  tests/smoke_curl.sh        # 本物の PG + サーバー相手の curl 通し
```


---
## まとめ

- HTTP は「テキストの往復」 — 6 ステップでフレームワーク無しで書けた
- DB は「整合性を強制してくれる表の集まり」 — 中間テーブルとトランザクションが武器
- Python から DB を触るのは「カーソル → 行 → コミット」 — psycopg も sqlite3 も骨格は同じ
- 認証はハッシュ + ソルト + 反復、セッションは Cookie でトークンを送るだけ
- ガチャ抽選は重み付き乱択 = 累積和 + 二分探索が古典実装、`random.choices` でも書ける
- 完走おつかれさま 🎉
